# DFG Re=20: L-BFGS continuation for the current RBF cohort

This private, internet-disabled T4 kernel consumes the private input dataset `rodrimandub/af-pikans-dfg-rbf-inputs-20260928`, where Kaggle has expanded the source files. It requires exactly one dataset mount, then validates the exact routes `rodrimandub/af-pikans-dfg-re20-baseline-20260927-r3` and `rodrimandub/af-pikans-dfg-re20-baseline-20260927-r4` directly inside that mount. Seeds 301–302 must come from the r3 route and seeds 303–305 from the r4 route; the loader checks each per-seed checkpoint and pool, the `run.json` and `provenance.json` seed ids, and the recorded pool SHA-256 before starting.

Each seed receives up to 540 seconds of deterministic full-pool L-BFGS. Two seeds run at once on the two Tesla T4 devices; the five seeds use waves 301/302, 303/304, then 305. The closure preserves the r4 Navier–Stokes residual and inlet, no-slip wall/cylinder, and natural outlet traction losses. Adam is not rerun. The training loss never uses a FEATFLOW error.

Before and after scores report FEATFLOW drag, lift, and front-minus-rear pressure at 512 and 1,024 cylinder points, plus held-out residual regions. The runtime writes source hashes, device assignments, elapsed times, optimizer states, and per-seed outputs to `/kaggle/working/dfg-current-cohort-refinement/`. `field_relative_l2` remains null and `scientific_accepted` remains false because no converged FEM field is packaged. This is an RBF-only cohort; it has no MLP arm.

The notebook has a 3,500-second work deadline, a 6,800-second watchdog, and a 7,200-second declared session cap. It requires exactly two visible Tesla T4 GPUs and fails if either device or either source route is missing or ambiguous.


In [ ]:
import base64
import csv
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import threading
import time

SESSION_STARTED = time.perf_counter()
QUOTA_CAP_SECONDS = 7200.0
SOFT_DEADLINE_SECONDS = 3500.0
WATCHDOG_SECONDS = 6800.0
DEADLINE = SESSION_STARTED + SOFT_DEADLINE_SECONDS
SOURCE_KERNELS = {
    "r3": "rodrimandub/af-pikans-dfg-re20-baseline-20260927-r3",
    "r4": "rodrimandub/af-pikans-dfg-re20-baseline-20260927-r4",
}
INPUT_DATASET_ID = "rodrimandub/af-pikans-dfg-rbf-inputs-20260928"
INPUT_DATASET_MOUNT_NAME = INPUT_DATASET_ID.rsplit("/", 1)[-1]
SOURCE_ROUTE_NAMES = {name: slug.rsplit("/", 1)[-1] for name, slug in SOURCE_KERNELS.items()}
SEEDS_BY_SOURCE = {"r3": (301, 302), "r4": (303, 304, 305)}
SEEDS = (301, 302, 303, 304, 305)
SEED_SOURCE = {seed: source for source, seeds in SEEDS_BY_SOURCE.items() for seed in seeds}
REQUIRED_SEED_FILES = (
    "dfg_rbf_pikan.pt",
    "dfg_training_pools.npz",
    "run.json",
    "provenance.json",
    "manifest_snapshot.json",
)
PAIR_SEED_SECONDS = 540.0
PAIR_WALL_LIMIT_SECONDS = 610.0
LBFGS_CHUNK_SIZE = 1024
HELDOUT_COUNT = 16384
HELDOUT_BOUNDARY_COUNT = 512

SESSION = Path("/kaggle/working/dfg-current-cohort-refinement") / ("run-" + str(time.time_ns()))
SESSION.mkdir(parents=True, exist_ok=False)
VENDOR_ROOT = SESSION / "vendor"
WORKER_PATH = SESSION / "dfg_rbf_lbfgs_worker.py"
SOURCE_MODULES = {'src/research_pipeline/sciml/dfg.py': '...', 'src/research_pipeline/sciml/dfg_campaign.py': '...'}
SOURCE_MODULE_HASHES = {'src/research_pipeline/sciml/dfg.py': '3aee5c92e1f05db0752eff3a57d4c995215cde6b781866d61eb99ebd7fcee7d4', 'src/research_pipeline/sciml/dfg_campaign.py': '1a6268624520dee7ee9aeadfe7d2c0bf23329427b54030fc821c7771130cbf13'}
WORKER_BYTES = base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJGdWxsLXBvb2wgTC1CRkdTIGNvbnRpbnVhdGlvbiBmb3Igb25lIHNhdmVkIERGRyBSQkYtUElLQU4gc2VlZC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCgpkZWYgc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShibG9jaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgd3JpdGVfanNvbihwYXRoOiBQYXRoLCBwYXlsb2FkOiBkaWN0KSAtPiBOb25lOgogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMocGF5bG9hZCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlLCBhbGxvd19uYW49RmFsc2UpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKCgpkZWYgY2xvbmVfc3RhdGVfdG9fY3B1KG1vZGVsOiB0b3JjaC5ubi5Nb2R1bGUpIC0+IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdOgogICAgcmV0dXJuIHtuYW1lOiB2YWx1ZS5kZXRhY2goKS5jcHUoKS5jbG9uZSgpIGZvciBuYW1lLCB2YWx1ZSBpbiBtb2RlbC5zdGF0ZV9kaWN0KCkuaXRlbXMoKX0KCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZHVsZS1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wb29sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLWtlcm5lbCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1jaGVja3BvaW50LXNoYTI1NiIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1wb29sLXNoYTI1NiIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1ydW4tc2hhMjU2IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLXByb3ZlbmFuY2Utc2hhMjU2IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlLW5hbWUiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1nbG9iYWwtZGV2aWNlLWluZGV4IiwgdHlwZT1pbnQsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlY29uZHMiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTU0MC4wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaHVuay1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAyNCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgaW1wb3J0IHN5cwoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoYXJncy5tb2R1bGVfcm9vdC5yZXNvbHZlKCkpKQogICAgZnJvbSBzcmMucmVzZWFyY2hfcGlwZWxpbmUuc2NpbWwuZGZnIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgICAgICBESUFNRVRFUiwKICAgICAgICBET01BSU5fWSwKICAgICAgICBOVSwKICAgICAgICBQUkVTU1VSRV9TQ0FMRSwKICAgICAgICBVX01FQU4sCiAgICAgICAgVV9NQVgsCiAgICAgICAgREZHUkJGUElLQU4sCiAgICAgICAgc3RlYWR5X25zX3Jlc2lkdWFsLAogICAgKQoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIG9yIHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoaXMgd29ya2VyIHJlcXVpcmVzIGV4YWN0bHkgb25lIHZpc2libGUgQ1VEQSBkZXZpY2U7IENQVSBmYWxsYmFjayBpcyBkaXNhYmxlZCIpCiAgICB2aXNpYmxlX25hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgaWYgIlQ0IiBub3QgaW4gdmlzaWJsZV9uYW1lLnVwcGVyKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRXhwZWN0ZWQgYSBUZXNsYSBUNCwgZm91bmQge3Zpc2libGVfbmFtZSFyfSIpCiAgICBpZiB2aXNpYmxlX25hbWUgIT0gYXJncy5kZXZpY2VfbmFtZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJBc3NpZ25lZCBHUFUgaWRlbnRpdHkgY2hhbmdlZDogZXhwZWN0ZWQge2FyZ3MuZGV2aWNlX25hbWUhcn0sIGZvdW5kIHt2aXNpYmxlX25hbWUhcn0iKQoKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcygyKQogICAgdG9yY2guYmFja2VuZHMuY3VkYS5tYXRtdWwuYWxsb3dfdGYzMiA9IEZhbHNlCiAgICB0b3JjaC5jdWRhLnNldF9kZXZpY2UoMCkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYTowIikKICAgIGFyZ3Mub3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdvcmtlcl9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQoKICAgIGNoZWNrcG9pbnRfc2hhID0gc2hhMjU2KGFyZ3MuY2hlY2twb2ludCkKICAgIHBvb2xfc2hhID0gc2hhMjU2KGFyZ3MucG9vbCkKICAgIGlmIGNoZWNrcG9pbnRfc2hhICE9IGFyZ3Muc291cmNlX2NoZWNrcG9pbnRfc2hhMjU2IG9yIHBvb2xfc2hhICE9IGFyZ3Muc291cmNlX3Bvb2xfc2hhMjU2OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiSW5wdXQgY2hlY2twb2ludCBvciBwb29sIGNoYW5nZWQgYWZ0ZXIgcHJvdmVuYW5jZSB2YWxpZGF0aW9uIikKCiAgICBjaGVja3BvaW50X3BheWxvYWQgPSB0b3JjaC5sb2FkKGFyZ3MuY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgaWYgbm90IGlzaW5zdGFuY2UoY2hlY2twb2ludF9wYXlsb2FkLCBkaWN0KSBvciBub3QgaXNpbnN0YW5jZShjaGVja3BvaW50X3BheWxvYWQuZ2V0KCJzdGF0ZV9kaWN0IiksIGRpY3QpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUkJGIGNoZWNrcG9pbnQgaGFzIG5vIHN0YXRlX2RpY3QiKQogICAgbW9kZWwgPSBERkdSQkZQSUtBTihzZWVkPWFyZ3Muc2VlZCkudG8oZGV2aWNlKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRfcGF5bG9hZFsic3RhdGVfZGljdCJdLCBzdHJpY3Q9VHJ1ZSkKICAgIG1vZGVsLnRyYWluKCkKCiAgICB3aXRoIG5wLmxvYWQoYXJncy5wb29sLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGFyY2hpdmU6CiAgICAgICAgcmVxdWlyZWQgPSAoImludGVyaW9yIiwgImlubGV0IiwgImxvd2VyX3dhbGwiLCAidXBwZXJfd2FsbCIsICJjeWxpbmRlciIsICJvdXRsZXQiKQogICAgICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiByZXF1aXJlZCBpZiBuYW1lIG5vdCBpbiBhcmNoaXZlLmZpbGVzXQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlNhdmVkIERGRyBwb29sIGlzIG1pc3NpbmcgYXJyYXlzOiB7bWlzc2luZ30iKQogICAgICAgIHBvb2xfYXJyYXlzID0ge25hbWU6IG5wLmFzYXJyYXkoYXJjaGl2ZVtuYW1lXSwgZHR5cGU9bnAuZmxvYXQ2NCkgZm9yIG5hbWUgaW4gcmVxdWlyZWR9CiAgICBmb3IgbmFtZSwgdmFsdWVzIGluIHBvb2xfYXJyYXlzLml0ZW1zKCk6CiAgICAgICAgaWYgdmFsdWVzLm5kaW0gIT0gMiBvciB2YWx1ZXMuc2hhcGVbMV0gIT0gMiBvciB2YWx1ZXMuc2hhcGVbMF0gPT0gMCBvciBub3QgbnAuaXNmaW5pdGUodmFsdWVzKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiU2F2ZWQgcG9vbCB7bmFtZSFyfSBtdXN0IGJlIGEgbm9uLWVtcHR5IGZpbml0ZSAobiwyKSBhcnJheSIpCiAgICB0ZW5zb3JzID0ge25hbWU6IHRvcmNoLmFzX3RlbnNvcih2YWx1ZXMsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1kZXZpY2UpIGZvciBuYW1lLCB2YWx1ZXMgaW4gcG9vbF9hcnJheXMuaXRlbXMoKX0KICAgIGludGVyaW9yX2NvdW50ID0gaW50KHRlbnNvcnNbImludGVyaW9yIl0uc2hhcGVbMF0pCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZGVhZGxpbmUgPSBzdGFydGVkICsgZmxvYXQoYXJncy5zZWNvbmRzKQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uTEJGR1MoCiAgICAgICAgbW9kZWwucGFyYW1ldGVycygpLAogICAgICAgIGxyPTEuMCwKICAgICAgICBtYXhfaXRlcj0xLAogICAgICAgIG1heF9ldmFsPTQsCiAgICAgICAgdG9sZXJhbmNlX2dyYWQ9MS4wZS04LAogICAgICAgIHRvbGVyYW5jZV9jaGFuZ2U9MS4wZS0xMCwKICAgICAgICBoaXN0b3J5X3NpemU9NTAsCiAgICAgICAgbGluZV9zZWFyY2hfZm49InN0cm9uZ193b2xmZSIsCiAgICApCiAgICBzdGVwcyA9IDAKICAgIGxhc3RfbG9zczogZmxvYXQgfCBOb25lID0gTm9uZQogICAgbGFzdF9jb21wb25lbnRzOiBkaWN0W3N0ciwgZmxvYXRdID0ge30KICAgIHN0b3BfcmVhc29uID0gInRpbWVfY2FwIgogICAgZXJyb3JfdGV4dDogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGhpc3Rvcnk6IGxpc3RbZGljdF0gPSBbXQogICAgYWNjZXB0ZWRfc3RhdGUgPSBjbG9uZV9zdGF0ZV90b19jcHUobW9kZWwpCiAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKGRldmljZSkKCiAgICBkZWYgY2xvc3VyZSgpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBub25sb2NhbCBsYXN0X2xvc3MsIGxhc3RfY29tcG9uZW50cwogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICB0b3RhbF92YWx1ZSA9IDAuMAogICAgICAgIGNvbXBvbmVudHMgPSB7InBkZV9tb21lbnR1bV94IjogMC4wLCAicGRlX21vbWVudHVtX3kiOiAwLjAsICJwZGVfY29udGludWl0eSI6IDAuMH0KCiAgICAgICAgZm9yIHN0YXJ0X2luZGV4IGluIHJhbmdlKDAsIGludGVyaW9yX2NvdW50LCBhcmdzLmNodW5rX3NpemUpOgogICAgICAgICAgICBpZiB0aW1lLnBlcmZfY291bnRlcigpID49IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKCJwZXItc2VlZCBMLUJGR1MgY2FwIHJlYWNoZWQgaW5zaWRlIGRldGVybWluaXN0aWMgZnVsbC1wb29sIGNsb3N1cmUiKQogICAgICAgICAgICBlbmRfaW5kZXggPSBtaW4oc3RhcnRfaW5kZXggKyBhcmdzLmNodW5rX3NpemUsIGludGVyaW9yX2NvdW50KQogICAgICAgICAgICB4eSA9IHRlbnNvcnNbImludGVyaW9yIl1bc3RhcnRfaW5kZXg6ZW5kX2luZGV4XS5kZXRhY2goKS5yZXF1aXJlc19ncmFkXyhUcnVlKQogICAgICAgICAgICByZXNpZHVhbHMgPSBzdGVhZHlfbnNfcmVzaWR1YWwobW9kZWwoeHkpLCB4eSkKICAgICAgICAgICAgd2VpZ2h0ID0gKGVuZF9pbmRleCAtIHN0YXJ0X2luZGV4KSAvIGludGVyaW9yX2NvdW50CiAgICAgICAgICAgIG5hbWVzID0gKCgibW9tZW50dW1feCIsICJwZGVfbW9tZW50dW1feCIpLCAoIm1vbWVudHVtX3kiLCAicGRlX21vbWVudHVtX3kiKSwgKCJjb250aW51aXR5IiwgInBkZV9jb250aW51aXR5IikpCiAgICAgICAgICAgIGNodW5rX3Rlcm1zID0gW10KICAgICAgICAgICAgZm9yIHJlc2lkdWFsX25hbWUsIGNvbXBvbmVudF9uYW1lIGluIG5hbWVzOgogICAgICAgICAgICAgICAgdmFsdWUgPSByZXNpZHVhbHNbcmVzaWR1YWxfbmFtZV0uc3F1YXJlKCkubWVhbigpICogd2VpZ2h0CiAgICAgICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcihmIm5vbi1maW5pdGUge2NvbXBvbmVudF9uYW1lfSBsb3NzIikKICAgICAgICAgICAgICAgIGNodW5rX3Rlcm1zLmFwcGVuZCh2YWx1ZSkKICAgICAgICAgICAgICAgIHNjYWxhciA9IGZsb2F0KHZhbHVlLmRldGFjaCgpLmNwdSgpKQogICAgICAgICAgICAgICAgY29tcG9uZW50c1tjb21wb25lbnRfbmFtZV0gKz0gc2NhbGFyCiAgICAgICAgICAgICAgICB0b3RhbF92YWx1ZSArPSBzY2FsYXIKICAgICAgICAgICAgY2h1bmtfbG9zcyA9IHRvcmNoLnN0YWNrKGNodW5rX3Rlcm1zKS5zdW0oKQogICAgICAgICAgICBjaHVua19sb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgZGVsIHh5LCByZXNpZHVhbHMKCiAgICAgICAgaW5sZXQgPSB0ZW5zb3JzWyJpbmxldCJdCiAgICAgICAgeSA9IGlubGV0WzosIDE6Ml0KICAgICAgICBpbmxldF91ID0gNC4wICogVV9NQVggKiB5ICogKERPTUFJTl9ZWzFdIC0geSkgLyAoRE9NQUlOX1lbMV0gLSBET01BSU5fWVswXSkgKiogMgogICAgICAgIGlubGV0X3RhcmdldCA9IHRvcmNoLmNhdCgoaW5sZXRfdSwgdG9yY2guemVyb3NfbGlrZShpbmxldF91KSksIGRpbT0xKQogICAgICAgIGJvdW5kYXJ5X3NldHMgPSAoCiAgICAgICAgICAgICgiaW5sZXQiLCBpbmxldF90YXJnZXQpLAogICAgICAgICAgICAoImxvd2VyX3dhbGwiLCB0b3JjaC56ZXJvc19saWtlKHRlbnNvcnNbImxvd2VyX3dhbGwiXSkpLAogICAgICAgICAgICAoInVwcGVyX3dhbGwiLCB0b3JjaC56ZXJvc19saWtlKHRlbnNvcnNbInVwcGVyX3dhbGwiXSkpLAogICAgICAgICAgICAoImN5bGluZGVyIiwgdG9yY2guemVyb3NfbGlrZSh0ZW5zb3JzWyJjeWxpbmRlciJdKSksCiAgICAgICAgKQogICAgICAgIGZvciBuYW1lLCB0YXJnZXQgaW4gYm91bmRhcnlfc2V0czoKICAgICAgICAgICAgdmFsdWUgPSAoKG1vZGVsKHRlbnNvcnNbbmFtZV0pWzosIDoyXSAtIHRhcmdldCkgLyBVX01FQU4pLnNxdWFyZSgpLm1lYW4oKQogICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYibm9uLWZpbml0ZSB7bmFtZX0gdmVsb2NpdHkgbG9zcyIpCiAgICAgICAgICAgIHZhbHVlLmJhY2t3YXJkKCkKICAgICAgICAgICAgc2NhbGFyID0gZmxvYXQodmFsdWUuZGV0YWNoKCkuY3B1KCkpCiAgICAgICAgICAgIGNvbXBvbmVudHNbZiJib3VuZGFyeV97bmFtZX0iXSA9IHNjYWxhcgogICAgICAgICAgICB0b3RhbF92YWx1ZSArPSBzY2FsYXIKCiAgICAgICAgb3V0bGV0ID0gdGVuc29yc1sib3V0bGV0Il0uZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICBmaWVsZHMgPSBtb2RlbChvdXRsZXQpCiAgICAgICAgZ3JhZF91ID0gdG9yY2guYXV0b2dyYWQuZ3JhZCgKICAgICAgICAgICAgZmllbGRzWzosIDBdLnN1bSgpLCBvdXRsZXQsIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZQogICAgICAgIClbMF0KICAgICAgICBncmFkX3YgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGZpZWxkc1s6LCAxXS5zdW0oKSwgb3V0bGV0LCBjcmVhdGVfZ3JhcGg9VHJ1ZSwgcmV0YWluX2dyYXBoPVRydWUpWzBdCiAgICAgICAgdHJhY3Rpb25feCA9IChOVSAqIGdyYWRfdVs6LCAwOjFdIC0gZmllbGRzWzosIDI6M10pIC8gUFJFU1NVUkVfU0NBTEUKICAgICAgICB0cmFjdGlvbl95ID0gTlUgKiBncmFkX3ZbOiwgMDoxXSAvIFBSRVNTVVJFX1NDQUxFCiAgICAgICAgb3V0bGV0X2xvc3MgPSB0cmFjdGlvbl94LnNxdWFyZSgpLm1lYW4oKSArIHRyYWN0aW9uX3kuc3F1YXJlKCkubWVhbigpCiAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKG91dGxldF9sb3NzKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCJub24tZmluaXRlIG91dGxldCB0cmFjdGlvbiBsb3NzIikKICAgICAgICBvdXRsZXRfbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgY29tcG9uZW50c1siYm91bmRhcnlfb3V0bGV0X3RyYWN0aW9uIl0gPSBmbG9hdChvdXRsZXRfbG9zcy5kZXRhY2goKS5jcHUoKSkKICAgICAgICB0b3RhbF92YWx1ZSArPSBjb21wb25lbnRzWyJib3VuZGFyeV9vdXRsZXRfdHJhY3Rpb24iXQogICAgICAgIGxhc3RfY29tcG9uZW50cyA9IGNvbXBvbmVudHMKICAgICAgICBsYXN0X2xvc3MgPSB0b3RhbF92YWx1ZQogICAgICAgIHJldHVybiB0b3JjaC5hc190ZW5zb3IodG90YWxfdmFsdWUsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1kZXZpY2UpCgogICAgdHJ5OgogICAgICAgIHdoaWxlIHRpbWUucGVyZl9jb3VudGVyKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgcHJpb3JfYWNjZXB0ZWQgPSBhY2NlcHRlZF9zdGF0ZQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcChjbG9zdXJlKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZShkZXZpY2UpCiAgICAgICAgICAgIGV4Y2VwdCBUaW1lb3V0RXJyb3I6CiAgICAgICAgICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QocHJpb3JfYWNjZXB0ZWQsIHN0cmljdD1UcnVlKQogICAgICAgICAgICAgICAgc3RvcF9yZWFzb24gPSAidGltZV9jYXBfZHVyaW5nX2Nsb3N1cmUiCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBleGNlcHQgRmxvYXRpbmdQb2ludEVycm9yIGFzIGV4YzoKICAgICAgICAgICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChwcmlvcl9hY2NlcHRlZCwgc3RyaWN0PVRydWUpCiAgICAgICAgICAgICAgICBzdG9wX3JlYXNvbiA9ICJub25maW5pdGVfbG9zcyIKICAgICAgICAgICAgICAgIGVycm9yX3RleHQgPSBzdHIoZXhjKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgYWNjZXB0ZWRfc3RhdGUgPSBjbG9uZV9zdGF0ZV90b19jcHUobW9kZWwpCiAgICAgICAgICAgIHN0ZXBzICs9IDEKICAgICAgICAgICAgaGlzdG9yeS5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInN0ZXAiOiBzdGVwcywKICAgICAgICAgICAgICAgICAgICAibG9zcyI6IGxhc3RfbG9zcywKICAgICAgICAgICAgICAgICAgICAiY29tcG9uZW50cyI6IGRpY3QobGFzdF9jb21wb25lbnRzKSwKICAgICAgICAgICAgICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgbGFzdF9sb3NzIGlzIE5vbmUgb3Igbm90IG1hdGguaXNmaW5pdGUobGFzdF9sb3NzKToKICAgICAgICAgICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChwcmlvcl9hY2NlcHRlZCwgc3RyaWN0PVRydWUpCiAgICAgICAgICAgICAgICBhY2NlcHRlZF9zdGF0ZSA9IHByaW9yX2FjY2VwdGVkCiAgICAgICAgICAgICAgICBzdG9wX3JlYXNvbiA9ICJub25maW5pdGVfbG9zcyIKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3RvcF9yZWFzb24gPSAidGltZV9jYXAiCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoYWNjZXB0ZWRfc3RhdGUsIHN0cmljdD1UcnVlKQogICAgICAgIHN0b3BfcmVhc29uID0gIm9wdGltaXplcl9lcnJvciIKICAgICAgICBlcnJvcl90ZXh0ID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCgogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZShkZXZpY2UpCiAgICBlbGFwc2VkID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKICAgIG91dHB1dF9jaGVja3BvaW50ID0gYXJncy5vdXRwdXQgLyAiY2hlY2twb2ludC1sYmZncy5wdCIKICAgIHRvcmNoLnNhdmUoCiAgICAgICAgewogICAgICAgICAgICAic3RhdGVfZGljdCI6IGFjY2VwdGVkX3N0YXRlLAogICAgICAgICAgICAic2VlZCI6IGFyZ3Muc2VlZCwKICAgICAgICAgICAgInN0YWdlIjogImRldGVybWluaXN0aWNfZnVsbF9wb29sX0xCRkdTX2NvbnRpbnVhdGlvbiIsCiAgICAgICAgICAgICJzb3VyY2VfY2hlY2twb2ludF9zaGEyNTYiOiBjaGVja3BvaW50X3NoYSwKICAgICAgICAgICAgInNvdXJjZV9wb29sX3NoYTI1NiI6IHBvb2xfc2hhLAogICAgICAgICAgICAibGJmZ3Nfc3RlcHMiOiBzdGVwcywKICAgICAgICAgICAgImxvc3NfY29tcG9uZW50cyI6IGxhc3RfY29tcG9uZW50cywKICAgICAgICB9LAogICAgICAgIG91dHB1dF9jaGVja3BvaW50LAogICAgKQogICAgd2l0aCAoYXJncy5vdXRwdXQgLyAibGJmZ3NfaGlzdG9yeS5qc29ubCIpLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIHJvdyBpbiBoaXN0b3J5OgogICAgICAgICAgICBzdHJlYW0ud3JpdGUoanNvbi5kdW1wcyhyb3csIGFsbG93X25hbj1GYWxzZSkgKyAiXG4iKQogICAgc3VtbWFyeSA9IHsKICAgICAgICAic3RhdHVzIjogImxiZmdzX3JlZmluZWQiIGlmIHN0ZXBzID4gMCBlbHNlIHN0b3BfcmVhc29uLAogICAgICAgICJzZWVkIjogYXJncy5zZWVkLAogICAgICAgICJzb3VyY2Vfa2VybmVsIjogYXJncy5zb3VyY2Vfa2VybmVsLAogICAgICAgICJzdGFnZSI6ICJkZXRlcm1pbmlzdGljX2Z1bGxfcG9vbF9MQkZHU19jb250aW51YXRpb24iLAogICAgICAgICJvcHRpbWl6ZXIiOiAidG9yY2gub3B0aW0uTEJGR1MiLAogICAgICAgICJwb29sX3BvbGljeSI6ICJzYXZlZCBmdWxsIHBvb2wsIGZpeGVkIG9yZGVyLCBkZXRlcm1pbmlzdGljIGNodW5rZWQgY2xvc3VyZSIsCiAgICAgICAgImxiZmdzX3N0ZXBzIjogc3RlcHMsCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyI6IGVsYXBzZWQsCiAgICAgICAgIndvcmtlcl93YWxsX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gd29ya2VyX3N0YXJ0ZWQsCiAgICAgICAgInNlY29uZHNfY2FwIjogYXJncy5zZWNvbmRzLAogICAgICAgICJzdG9wX3JlYXNvbiI6IHN0b3BfcmVhc29uLAogICAgICAgICJsYXN0X2xvc3MiOiBsYXN0X2xvc3MsCiAgICAgICAgImxvc3NfY29tcG9uZW50cyI6IGxhc3RfY29tcG9uZW50cywKICAgICAgICAiZGV2aWNlIjogdmlzaWJsZV9uYW1lLAogICAgICAgICJjdWRhX2luZGV4X3Zpc2libGVfdG9fd29ya2VyIjogMCwKICAgICAgICAiZ2xvYmFsX2RldmljZV9pbmRleCI6IGFyZ3MuZ2xvYmFsX2RldmljZV9pbmRleCwKICAgICAgICAiaW50ZXJpb3JfcG9pbnRzIjogaW50ZXJpb3JfY291bnQsCiAgICAgICAgImJvdW5kYXJ5X3BvaW50c19wZXJfc2V0Ijoge25hbWU6IGludCh0ZW5zb3JzW25hbWVdLnNoYXBlWzBdKSBmb3IgbmFtZSBpbiAoImlubGV0IiwgImxvd2VyX3dhbGwiLCAidXBwZXJfd2FsbCIsICJjeWxpbmRlciIsICJvdXRsZXQiKX0sCiAgICAgICAgImNodW5rX3NpemUiOiBhcmdzLmNodW5rX3NpemUsCiAgICAgICAgImlucHV0X2NoZWNrcG9pbnRfc2hhMjU2IjogY2hlY2twb2ludF9zaGEsCiAgICAgICAgImlucHV0X3Bvb2xfc2hhMjU2IjogcG9vbF9zaGEsCiAgICAgICAgInNvdXJjZV9ydW5fc2hhMjU2IjogYXJncy5zb3VyY2VfcnVuX3NoYTI1NiwKICAgICAgICAic291cmNlX3Byb3ZlbmFuY2Vfc2hhMjU2IjogYXJncy5zb3VyY2VfcHJvdmVuYW5jZV9zaGEyNTYsCiAgICAgICAgIm91dHB1dF9jaGVja3BvaW50Ijogc3RyKG91dHB1dF9jaGVja3BvaW50KSwKICAgICAgICAib3V0cHV0X2NoZWNrcG9pbnRfc2hhMjU2Ijogc2hhMjU2KG91dHB1dF9jaGVja3BvaW50KSwKICAgICAgICAicGVha19jdWRhX21lbW9yeV9hbGxvY2F0ZWRfYnl0ZXMiOiBpbnQodG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZChkZXZpY2UpKSwKICAgICAgICAiZmllbGRfcmVsYXRpdmVfbDIiOiBOb25lLAogICAgICAgICJzY2llbnRpZmljX2FjY2VwdGVkIjogRmFsc2UsCiAgICB9CiAgICBpZiBlcnJvcl90ZXh0IGlzIG5vdCBOb25lOgogICAgICAgIHN1bW1hcnlbImVycm9yIl0gPSBlcnJvcl90ZXh0CiAgICB3cml0ZV9qc29uKGFyZ3Mub3V0cHV0IC8gImxiZmdzX3N1bW1hcnkuanNvbiIsIHN1bW1hcnkpCiAgICBwcmludChqc29uLmR1bXBzKHN1bW1hcnksIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=")


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def require_t4x2(torch):
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("Expected exactly two CUDA devices; CPU fallback is disabled")
    names = [torch.cuda.get_device_name(index) for index in range(2)]
    if any("T4" not in name.upper() for name in names):
        raise RuntimeError(f"Expected two Tesla T4 devices; found {names!r}")
    return names


def install_official_modules():
    module_hashes = {}
    for relative_path, encoded in SOURCE_MODULES.items():
        payload = base64.b64decode(encoded)
        destination = VENDOR_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(payload)
        module_hashes[relative_path] = sha256(destination)
        if module_hashes[relative_path] != SOURCE_MODULE_HASHES[relative_path]:
            raise RuntimeError(f"Embedded official module hash mismatch: {relative_path}")
    for relative_path in (
        "src/__init__.py",
        "src/research_pipeline/__init__.py",
        "src/research_pipeline/sciml/__init__.py",
    ):
        destination = VENDOR_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text("", encoding="utf-8")
    WORKER_PATH.write_bytes(WORKER_BYTES)
    return module_hashes


def json_file(path, label):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception as exc:
        raise RuntimeError(f"Could not parse {label} at {path}: {type(exc).__name__}: {exc}") from exc


def seed_file_candidates(mount, seed, filename):
    wanted_parent = f"seed-{seed}"
    return sorted(
        path.resolve()
        for path in Path(mount).rglob(filename)
        if path.is_file() and path.parent.name == wanted_parent
    )


def seed_checkpoint_files(mount):
    found = {}
    for path in Path(mount).rglob("dfg_rbf_pikan.pt"):
        if path.is_file() and path.parent.name.startswith("seed-"):
            suffix = path.parent.name.removeprefix("seed-")
            if suffix.isdigit():
                found.setdefault(int(suffix), []).append(path.resolve())
    return found


def find_exact_source_mounts(input_root):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise FileNotFoundError("Kaggle input mount /kaggle/input is missing")
    dataset_mounts = [
        path.resolve()
        for path in input_root.iterdir()
        if path.is_dir() and path.name.casefold() == INPUT_DATASET_MOUNT_NAME.casefold()
    ]
    if len(dataset_mounts) != 1:
        raise RuntimeError(
            json.dumps(
                {
                    "message": "Expected exactly one mounted private source dataset",
                    "dataset_id": INPUT_DATASET_ID,
                    "expected_mount_name": INPUT_DATASET_MOUNT_NAME,
                    "matching_mounts": [str(path) for path in dataset_mounts],
                },
                sort_keys=True,
            )
        )
    dataset_root = dataset_mounts[0]

    def is_within(path, root):
        try:
            return os.path.commonpath((str(Path(path).resolve()), str(Path(root).resolve()))) == str(Path(root).resolve())
        except ValueError:
            return False

    def route_candidates(search_root, source_name):
        owner = SOURCE_KERNELS[source_name].split("/", 1)[0]
        route_name = SOURCE_ROUTE_NAMES[source_name]
        matches = []
        for current, directory_names, _file_names in os.walk(search_root):
            if Path(current).name.casefold() != owner.casefold():
                continue
            matches.extend(
                (Path(current) / directory_name).resolve()
                for directory_name in directory_names
                if directory_name.casefold() == route_name.casefold()
            )
        return sorted(matches)

    selected = {}
    for source_name, expected_route_name in SOURCE_ROUTE_NAMES.items():
        stray_matches = route_candidates(input_root, source_name)
        matches = route_candidates(dataset_root, source_name)
        if len(stray_matches) != 1 or len(matches) != 1 or stray_matches[0] != matches[0]:
            raise RuntimeError(
                json.dumps(
                    {
                        "message": "Expected exactly one exact source-kernel route inside the configured dataset",
                        "dataset_id": INPUT_DATASET_ID,
                        "dataset_mount": str(dataset_root),
                        "source": source_name,
                        "kernel_slug": SOURCE_KERNELS[source_name],
                        "expected_route_name": expected_route_name,
                        "matching_routes_in_dataset": [str(path) for path in matches],
                        "matching_routes_under_input": [str(path) for path in stray_matches],
                    },
                    sort_keys=True,
                )
            )
        selected[source_name] = matches[0]

    for current, _directory_names, file_names in os.walk(input_root):
        if "dfg_rbf_pikan.pt" not in file_names:
            continue
        checkpoint = Path(current) / "dfg_rbf_pikan.pt"
        parent_name = checkpoint.parent.name
        if not parent_name.startswith("seed-") or not parent_name.removeprefix("seed-").isdigit():
            continue
        seed = int(parent_name.removeprefix("seed-"))
        if seed not in SEED_SOURCE:
            continue
        expected_root = selected[SEED_SOURCE[seed]]
        if not is_within(checkpoint, expected_root):
            raise RuntimeError(
                f"Target seed {seed} checkpoint appears outside its exact source route "
                f"{SOURCE_KERNELS[SEED_SOURCE[seed]]}: {checkpoint}"
            )
    return dataset_root, selected


def validate_source_seed(source_name, mount, seed):
    candidates = seed_file_candidates(mount, seed, "dfg_rbf_pikan.pt")
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one checkpoint for seed {seed} under {SOURCE_KERNELS[source_name]}; found {len(candidates)}"
        )
    seed_dir = candidates[0].parent
    files = {}
    for filename in REQUIRED_SEED_FILES:
        matches = seed_file_candidates(mount, seed, filename)
        if len(matches) != 1 or matches[0].parent != seed_dir:
            raise RuntimeError(
                f"Expected one {filename} beside seed {seed} checkpoint under {SOURCE_KERNELS[source_name]}; "
                f"found {[str(path) for path in matches]}"
            )
        files[filename] = matches[0]

    run = json_file(files["run.json"], "run.json")
    provenance = json_file(files["provenance.json"], "provenance.json")
    manifest = json_file(files["manifest_snapshot.json"], "manifest_snapshot.json")
    try:
        run_seed = int(run["seed"])
        training_seed = int(run["training_config"]["seed"])
        provenance_seed = int(provenance["sampling_seed_stream"]["training"])
    except (KeyError, TypeError, ValueError) as exc:
        raise RuntimeError(f"Missing or invalid run/provenance seed id for seed {seed}") from exc
    if run_seed != seed or training_seed != seed or provenance_seed != seed:
        raise RuntimeError(
            f"Seed identity mismatch at {seed_dir}: run.seed={run_seed}, training.seed={training_seed}, "
            f"provenance.training={provenance_seed}"
        )
    if run.get("problem") != "dfg_2d_1_steady_re20" or provenance.get("problem_id") != "dfg_2d_1_steady_re20":
        raise RuntimeError(f"Seed {seed} is not the expected DFG Re=20 problem")
    expected_model = "dfg_rbf_fixed_2_16_16_3"
    run_model = run.get("model", {})
    provenance_model = provenance.get("model", {})
    if run_model.get("id") != expected_model or provenance_model.get("id") != expected_model:
        raise RuntimeError(f"Seed {seed} does not identify the expected r4 RBF-PIKAN model")
    if run_model.get("architecture") != [2, 16, 16, 3] or provenance_model.get("architecture") != [2, 16, 16, 3]:
        raise RuntimeError(f"Seed {seed} architecture does not match [2,16,16,3]")
    if int(run_model.get("basis_count", -1)) != 11 or int(provenance_model.get("basis_count", -1)) != 11:
        raise RuntimeError(f"Seed {seed} does not use 11 RBF bases")

    checkpoint_hash = sha256(files["dfg_rbf_pikan.pt"])
    pool_hash = sha256(files["dfg_training_pools.npz"])
    run_pool_hash = run.get("pool_archive_sha256")
    if run.get("pool_archive") != "dfg_training_pools.npz" or run_pool_hash != pool_hash:
        raise RuntimeError(f"Seed {seed} saved-pool SHA-256 does not match run.json")
    file_hashes = {filename: sha256(path) for filename, path in files.items()}
    return {
        "seed": seed,
        "source": source_name,
        "source_kernel": SOURCE_KERNELS[source_name],
        "source_mount": str(Path(mount).resolve()),
        "seed_directory": str(seed_dir),
        "files": {filename: str(path) for filename, path in files.items()},
        "sha256": file_hashes,
        "checkpoint_sha256": checkpoint_hash,
        "pool_sha256": pool_hash,
        "run_seed": run_seed,
        "provenance_training_seed": provenance_seed,
        "manifest_model_id": manifest.get("model", {}).get("id"),
    }


def discover_sources():
    dataset_root, selected_mounts = find_exact_source_mounts("/kaggle/input")
    source_by_seed = {}
    for source_name, mount in selected_mounts.items():
        found = seed_checkpoint_files(mount)
        expected_seeds = set(SEEDS_BY_SOURCE[source_name])
        present_target_seeds = set(found).intersection(SEEDS)
        if present_target_seeds != expected_seeds:
            raise RuntimeError(
                f"Source {SOURCE_KERNELS[source_name]} must contain exactly target seeds {sorted(expected_seeds)}; "
                f"found target seeds {sorted(present_target_seeds)}"
            )
        for seed in SEEDS_BY_SOURCE[source_name]:
            if seed in source_by_seed:
                raise RuntimeError(f"Seed {seed} maps to more than one source kernel")
            source_by_seed[seed] = validate_source_seed(source_name, mount, seed)
    if set(source_by_seed) != set(SEEDS):
        raise RuntimeError(f"The source loader did not resolve exactly the required seed set {list(SEEDS)}")
    for seed, source in SEED_SOURCE.items():
        if source_by_seed[seed]["source"] != source:
            raise RuntimeError(f"Seed {seed} resolved to {source_by_seed[seed]['source']}, expected {source}")
    return dataset_root, selected_mounts, source_by_seed


def prepare_heldout(dfg_module, seed):
    heldout_seed = (int(seed) + 100000) % (2**32 - 1)
    pools, provenance = dfg_module.generate_dfg_pools(
        heldout_seed,
        interior_count=HELDOUT_COUNT,
        boundary_count=HELDOUT_BOUNDARY_COUNT,
        return_provenance=True,
    )
    points = pools["interior"]
    return {
        "seed": heldout_seed,
        "points": points,
        "points_sha256": provenance["pool_sha256"]["interior"],
        "pool_provenance": provenance,
        "origin": "official_r4_scrambled_sobol_generator_reconstructed_from_heldout_seed",
    }


def load_model(dfg_module, source_record, checkpoint_path, device):
    seed = int(source_record["seed"])
    payload = __import__("torch").load(checkpoint_path, map_location=device, weights_only=False)
    if not isinstance(payload, dict) or not isinstance(payload.get("state_dict"), dict):
        raise RuntimeError(f"Checkpoint for seed {seed} has no state_dict")
    if "output_scales" not in payload["state_dict"]:
        raise RuntimeError(f"Checkpoint for seed {seed} lacks the r4 output_scales buffer")
    saved_run = payload.get("run", {})
    saved_seed = saved_run.get("seed") if isinstance(saved_run, dict) else None
    if saved_seed is not None and int(saved_seed) != seed:
        raise RuntimeError(f"Checkpoint payload seed {saved_seed} does not match seed directory {seed}")
    model = dfg_module.DFGRBFPIKAN(seed=seed).to(device)
    model.load_state_dict(payload["state_dict"], strict=True)
    model.eval()
    return model


def score_checkpoint(dfg_module, campaign_module, source_record, checkpoint_path, stage, device, heldout):
    import torch

    seed = int(source_record["seed"])
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device)
    model = load_model(dfg_module, source_record, checkpoint_path, device)
    qoi512 = dfg_module.evaluate_dfg_qois(model, device=device, surface_count=512)
    qoi1024 = dfg_module.evaluate_dfg_qois(model, device=device, surface_count=1024)
    residual_arrays, residual_seconds = campaign_module._evaluate_heldout(
        model, device, heldout["points"], chunk_size=1024
    )
    residual_regions = campaign_module.summarize_residual_regions(
        heldout["points"], residual_arrays, near_distance=0.15
    )
    reference = dict(dfg_module.FEATFLOW_REFERENCE)
    errors = {
        "drag_relative_error": abs(qoi512["drag_coefficient"] - reference["drag_coefficient"])
        / reference["drag_coefficient"],
        "lift_absolute_error": abs(qoi512["lift_coefficient"] - reference["lift_coefficient"]),
        "pressure_difference_relative_error": abs(qoi512["pressure_difference"] - reference["pressure_difference"])
        / reference["pressure_difference"],
    }
    quadrature_change = {
        key: abs(qoi1024[key] - qoi512[key])
        for key in ("drag_coefficient", "lift_coefficient", "pressure_difference")
    }
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    result = {
        "status": "scored",
        "seed": seed,
        "source": source_record["source"],
        "source_kernel": source_record["source_kernel"],
        "stage": stage,
        "checkpoint": str(Path(checkpoint_path).resolve()),
        "checkpoint_sha256": sha256(checkpoint_path),
        "qois_512": qoi512,
        "qois_1024": qoi1024,
        "quadrature_512_to_1024_absolute_change": quadrature_change,
        "featflow_reference": reference,
        "qoi_errors_vs_featflow": errors,
        "heldout_residual_regions": residual_regions,
        "heldout_residual_seconds": residual_seconds,
        "heldout_point_count": int(len(heldout["points"])),
        "heldout_seed": heldout["seed"],
        "heldout_points_sha256": heldout["points_sha256"],
        "heldout_points_origin": heldout["origin"],
        "device": torch.cuda.get_device_name(device),
        "cuda_device_index": int(device.index),
        "peak_cuda_memory_allocated_bytes": int(torch.cuda.max_memory_allocated(device)),
        "elapsed_seconds": time.perf_counter() - started,
        "field_relative_l2": None,
        "scientific_accepted": False,
    }
    del model, residual_arrays
    torch.cuda.empty_cache()
    return result


def launch_wave(wave_number, seeds, source_by_seed, worker_path, module_root, gpu_names, train_root):
    children = []
    started = time.perf_counter()
    launch_deadline = min(started + PAIR_WALL_LIMIT_SECONDS, DEADLINE - 180.0)
    for offset, seed in enumerate(seeds):
        device_index = offset
        source = source_by_seed[seed]
        out = train_root / f"seed-{seed}"
        out.mkdir(parents=True, exist_ok=True)
        files = source["files"]
        command = [
            sys.executable,
            str(worker_path),
            "--module-root",
            str(module_root),
            "--checkpoint",
            files["dfg_rbf_pikan.pt"],
            "--pool",
            files["dfg_training_pools.npz"],
            "--output",
            str(out),
            "--seed",
            str(seed),
            "--source-kernel",
            source["source_kernel"],
            "--source-checkpoint-sha256",
            source["checkpoint_sha256"],
            "--source-pool-sha256",
            source["pool_sha256"],
            "--source-run-sha256",
            source["sha256"]["run.json"],
            "--source-provenance-sha256",
            source["sha256"]["provenance.json"],
            "--device-name",
            gpu_names[device_index],
            "--global-device-index",
            str(device_index),
            "--seconds",
            str(PAIR_SEED_SECONDS),
            "--chunk-size",
            str(LBFGS_CHUNK_SIZE),
        ]
        environment = os.environ.copy()
        environment["CUDA_VISIBLE_DEVICES"] = str(device_index)
        log_path = out / "worker.log"
        log = log_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, env=environment, stdout=log, stderr=subprocess.STDOUT)
        children.append(
            {
                "seed": seed,
                "source": source["source"],
                "device_index": device_index,
                "device_name": gpu_names[device_index],
                "output": out,
                "log_path": log_path,
                "log": log,
                "process": process,
                "started_offset_seconds": time.perf_counter() - started,
            }
        )

    timed_out = False
    while any(child["process"].poll() is None for child in children):
        if time.perf_counter() >= launch_deadline:
            timed_out = True
            for child in children:
                if child["process"].poll() is None:
                    child["process"].terminate()
            for child in children:
                process = child["process"]
                if process.poll() is None:
                    try:
                        process.wait(timeout=8)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait(timeout=8)
            break
        time.sleep(1.0)

    outcomes = {}
    for child in children:
        process = child["process"]
        return_code = process.poll()
        child["log"].close()
        summary_path = child["output"] / "lbfgs_summary.json"
        if summary_path.is_file():
            summary = json_file(summary_path, "L-BFGS worker summary")
        else:
            summary = {
                "status": "worker_failed_or_wave_timeout",
                "seed": child["seed"],
                "source": child["source"],
                "device": child["device_name"],
                "global_device_index": child["device_index"],
                "returncode": return_code,
                "wave_timed_out": timed_out,
                "log_tail": child["log_path"].read_text(encoding="utf-8", errors="replace")[-5000:],
                "elapsed_seconds": time.perf_counter() - started,
                "field_relative_l2": None,
                "scientific_accepted": False,
            }
            write_json(child["output"] / "failure.json", summary)
        summary["wave"] = wave_number
        summary["device"] = summary.get("device", child["device_name"])
        summary["global_device_index"] = child["device_index"]
        summary["process_returncode"] = return_code
        summary["wave_timed_out"] = timed_out
        summary["status"] = "lbfgs_refined" if return_code == 0 and int(summary.get("lbfgs_steps", 0)) > 0 else summary.get("status", "worker_failed")
        write_json(summary_path, summary)
        outcomes[child["seed"]] = summary
    return outcomes, time.perf_counter() - started


def main():
    import numpy as np
    import scipy
    import torch

    watchdog = threading.Timer(WATCHDOG_SECONDS, lambda: os.kill(os.getpid(), signal.SIGTERM))
    watchdog.daemon = True
    watchdog.start()
    try:
        module_hashes = install_official_modules()
        GPU_NAMES = require_t4x2(torch)
        torch.set_num_threads(2)
        torch.set_default_dtype(torch.float32)
        torch.backends.cuda.matmul.allow_tf32 = False
        sys.path.insert(0, str(VENDOR_ROOT))
        from src.research_pipeline.sciml import dfg as dfg_module
        from src.research_pipeline.sciml import dfg_campaign as campaign_module

        dataset_root, mounts, source_by_seed = discover_sources()
        write_json(
            SESSION / "source-validation.json",
            {
                "status": "validated",
                "input_dataset_id": INPUT_DATASET_ID,
                "input_dataset_mount": str(dataset_root),
                "source_kernels": SOURCE_KERNELS,
                "source_mounts": {key: str(path.resolve()) for key, path in mounts.items()},
                "seeds_by_source": SEEDS_BY_SOURCE,
                "source_by_seed": source_by_seed,
                "official_module_sha256": module_hashes,
                "gpu_names": GPU_NAMES,
                "python": sys.version,
                "numpy": np.__version__,
                "scipy": scipy.__version__,
                "torch": torch.__version__,
            },
        )
        heldout_by_seed = {seed: prepare_heldout(dfg_module, seed) for seed in SEEDS}
        before_scores = {}
        after_scores = {}
        for seed in SEEDS:
            if time.perf_counter() >= DEADLINE - 220.0:
                break
            score = score_checkpoint(
                dfg_module,
                campaign_module,
                source_by_seed[seed],
                source_by_seed[seed]["files"]["dfg_rbf_pikan.pt"],
                "adam_checkpoint_before_lbfgs",
                torch.device("cuda:0"),
                heldout_by_seed[seed],
            )
            before_scores[seed] = score
            write_json(SESSION / "scores-before" / f"seed-{seed}.json", score)

        train_root = SESSION / "training"
        train_root.mkdir(parents=True, exist_ok=True)
        worker_summaries = {}
        wave_summaries = []
        waves = [SEEDS[index : index + 2] for index in range(0, len(SEEDS), 2)]
        for wave_number, wave in enumerate(waves, start=1):
            if time.perf_counter() >= DEADLINE - 800.0:
                for seed in wave:
                    worker_summaries[seed] = {
                        "status": "not_started_soft_deadline",
                        "seed": seed,
                        "source": SEED_SOURCE[seed],
                        "field_relative_l2": None,
                        "scientific_accepted": False,
                    }
                continue
            outcomes, wall_seconds = launch_wave(
                wave_number,
                wave,
                source_by_seed,
                WORKER_PATH,
                VENDOR_ROOT,
                GPU_NAMES,
                train_root,
            )
            worker_summaries.update(outcomes)
            wave_summaries.append(
                {
                    "wave": wave_number,
                    "seeds": list(wave),
                    "wall_seconds": wall_seconds,
                    "workers": {str(seed): outcomes.get(seed) for seed in wave},
                }
            )
            write_json(SESSION / "training" / f"wave-{wave_number}.json", wave_summaries[-1])

        for seed in SEEDS:
            training = worker_summaries.get(seed, {})
            checkpoint_path = train_root / f"seed-{seed}" / "checkpoint-lbfgs.pt"
            if (
                int(training.get("lbfgs_steps", 0)) > 0
                and checkpoint_path.is_file()
                and time.perf_counter() < DEADLINE - 100.0
            ):
                score = score_checkpoint(
                    dfg_module,
                    campaign_module,
                    source_by_seed[seed],
                    checkpoint_path,
                    "lbfgs_refined",
                    torch.device("cuda:0"),
                    heldout_by_seed[seed],
                )
                score["qoi_delta_vs_before"] = {
                    key: score["qois_512"][key] - before_scores[seed]["qois_512"][key]
                    for key in ("drag_coefficient", "lift_coefficient", "pressure_difference")
                } if seed in before_scores else None
                after_scores[seed] = score
                write_json(SESSION / "scores-after" / f"seed-{seed}.json", score)

        results = []
        for seed in SEEDS:
            training = worker_summaries.get(
                seed,
                {"status": "not_started", "seed": seed, "source": SEED_SOURCE[seed]},
            )
            before = before_scores.get(seed)
            after = after_scores.get(seed)
            status = "complete" if before is not None and after is not None and int(training.get("lbfgs_steps", 0)) > 0 else "partial"
            results.append(
                {
                    "seed": seed,
                    "source": SEED_SOURCE[seed],
                    "source_kernel": SOURCE_KERNELS[SEED_SOURCE[seed]],
                    "source_artifacts": source_by_seed[seed],
                    "training": training,
                    "before": before,
                    "after": after,
                    "status": status,
                    "field_relative_l2": None,
                    "scientific_accepted": False,
                }
            )

        columns = (
            "seed",
            "source",
            "status",
            "lbfgs_steps",
            "training_elapsed_seconds",
            "device",
            "before_drag_coefficient",
            "after_drag_coefficient",
            "before_lift_coefficient",
            "after_lift_coefficient",
            "before_pressure_difference",
            "after_pressure_difference",
            "field_relative_l2",
            "scientific_accepted",
        )
        with (SESSION / "dfg-current-cohort-refinement.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
            writer.writeheader()
            for row in results:
                before = row.get("before") or {}
                after = row.get("after") or {}
                writer.writerow(
                    {
                        "seed": row["seed"],
                        "source": row["source"],
                        "status": row["status"],
                        "lbfgs_steps": row.get("training", {}).get("lbfgs_steps"),
                        "training_elapsed_seconds": row.get("training", {}).get("elapsed_seconds"),
                        "device": row.get("training", {}).get("device"),
                        "before_drag_coefficient": (before.get("qois_512") or {}).get("drag_coefficient"),
                        "after_drag_coefficient": (after.get("qois_512") or {}).get("drag_coefficient"),
                        "before_lift_coefficient": (before.get("qois_512") or {}).get("lift_coefficient"),
                        "after_lift_coefficient": (after.get("qois_512") or {}).get("lift_coefficient"),
                        "before_pressure_difference": (before.get("qois_512") or {}).get("pressure_difference"),
                        "after_pressure_difference": (after.get("qois_512") or {}).get("pressure_difference"),
                        "field_relative_l2": None,
                        "scientific_accepted": False,
                    }
                )

        completed = sum(row["status"] == "complete" for row in results)
        report = {
            "status": "lbfgs_refinement_complete" if completed == len(SEEDS) else "lbfgs_refinement_partial",
            "model_family": "RBF-PIKAN only",
            "input_dataset_id": INPUT_DATASET_ID,
            "input_dataset_mount": str(dataset_root),
            "source_kernels": SOURCE_KERNELS,
            "seeds_by_source": SEEDS_BY_SOURCE,
            "seeds": list(SEEDS),
            "lbfgs_seconds_per_seed_cap": PAIR_SEED_SECONDS,
            "lbfgs_chunk_size": LBFGS_CHUNK_SIZE,
            "training_waves": wave_summaries,
            "source_mounts": {key: str(path.resolve()) for key, path in mounts.items()},
            "source_by_seed": source_by_seed,
            "official_module_sha256": module_hashes,
            "gpu_names": GPU_NAMES,
            "quota_cap_seconds": QUOTA_CAP_SECONDS,
            "soft_deadline_seconds": SOFT_DEADLINE_SECONDS,
            "watchdog_seconds_from_first_code_cell": WATCHDOG_SECONDS,
            "session_wall_seconds_from_first_code_cell": time.perf_counter() - SESSION_STARTED,
            "successful_refined_and_scored_seeds": completed,
            "expected_seeds": len(SEEDS),
            "results": results,
            "field_relative_l2": None,
            "field_reference_status": "missing_converged_FEM_field",
            "scientific_accepted": False,
        }
        write_json(SESSION / "dfg-current-cohort-refinement.json", report)
        print(
            json.dumps(
                {
                    "status": report["status"],
                    "refined_and_scored": completed,
                    "session": str(SESSION),
                    "wall_seconds": report["session_wall_seconds_from_first_code_cell"],
                    "quota_cap_seconds": QUOTA_CAP_SECONDS,
                    "watchdog_seconds": WATCHDOG_SECONDS,
                    "scientific_accepted": False,
                },
                indent=2,
            )
        )
    finally:
        watchdog.cancel()


main()
